# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset using the `mlcroissant` library. We'll walk through data loading, schema and field discovery, extraction, preliminary EDA, and simple visualization, referencing all data elements by their `@id`s.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is a single object
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets, along with their fields and the `@id`s.

We identify all record sets and then, for each, enumerate its field `@id`s, following the Croissant schema conventions.

In [ ]:
# Review available record sets and their fields by @id
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the dataset schema.")
else:
    for rs in record_sets:
        print(f"RecordSet name: {rs.name}, @id: {rs.id}")
        if rs.fields:
            print("  Fields:")
            for field in rs.fields:
                print(f"    - {field.name} (@id: {field.id}) | DataType: {field.data_type}")
        else:
            print("  No fields defined.")
        print()

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. All record set and field references use their `@id`s.

In [ ]:
# Extract all records from each record set using @id as the key
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]  # get all record set @id's

if not record_set_ids:
    print("No record sets to extract.")
else:
    for record_set_id in record_set_ids:
        # Extract all records for the given record set
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from RecordSet @id: {record_set_id}")
    # Print columns of the first available recordset
    first_rs_id = record_set_ids[0]
    print(f"\nColumns in first record set (@id: {first_rs_id}):")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering numeric fields, normalizing, and grouping. 
We select candidate numeric and group fields from the record set's field list by their `@id`.

In [ ]:
# For EDA, select a numeric field (@id) and a group field (@id) to operate on.

if dataframes:
    # Use first record set as example
    rs = dataset.record_sets[0]
    df = dataframes[rs.id]
    numeric_field = None
    group_field = None
    # Find a field with data_type float or integer
    for field in rs.fields:
        if field.data_type in ('Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number'):
            if field.id in df.columns:
                numeric_field = field.id
                break
    # Try to find a categorical/grouping field
    for field in rs.fields:
        if field.data_type in ('Text', 'schema:Text', 'String') and field.id in df.columns:
            group_field = field.id
            break
    if numeric_field is not None:
        print(f"Using numeric field (@id): {numeric_field}")
        threshold = 10
        # Remove missing/non-numeric as needed
        filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
        ) / filtered_df[numeric_field].astype(float).std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field is not None and group_field in filtered_df.columns:
            # Only group by categorical fields that are not too numerous
            print(f"Grouping by field (@id): {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found in the record set for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field or relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    df_numeric = pd.to_numeric(df[numeric_field], errors='coerce')
    sns.histplot(df_numeric.dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(f"{numeric_field}")
    plt.ylabel("Count")
    plt.show()

    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Unable to plot: missing numeric field.")

## 6. Conclusion
We explored the FAIR² dataset via its Croissant schema using `mlcroissant`, loading its metadata, inspecting available record sets and their fields by `@id`, extracting data, and performing basic EDA and visualizations. For more detailed analysis, update the notebook to select specific field and record set `@id`s suited to your research questions.
